# Week 5: TensorFlow Acceleration and Interoperability
### NPTEL – Applied Accelerated Artificial Intelligence
**Instructor:** Dr. Satyajit Das, IIT Guwahati

---

## Course Overview
This notebook covers all five lectures of Week 5:

| Lecture | Topic |
|---------|-------|
| 1 | TensorFlow Execution Models and Performance Basics |
| 2 | Optimizing Input Pipelines with tf.data |
| 3 | Graph Execution and TensorFlow Optimization Techniques |
| 4 | Introduction to XLA Compilation (Demo) |
| 5 | Model Interoperability: SavedModel vs ONNX |

**Prerequisites:** Python 3.8+, TensorFlow 2.x  
**Install:** `pip install tensorflow numpy matplotlib onnx onnxruntime tf2onnx`

In [1]:
# ── Environment Setup ──────────────────────────────────────────────────
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'       # suppress TF info messages
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'      # deterministic results

import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')                           # non-interactive backend (change to 'inline' in Jupyter)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import numpy as np
import time
import tempfile
import shutil

import tensorflow as tf

print(f"TensorFlow  : {tf.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"Eager mode  : {tf.executing_eagerly()}")
print(f"GPU devices : {tf.config.list_physical_devices('GPU')}")
print(f"CPU devices : {len(tf.config.list_physical_devices('CPU'))}")

tf.random.set_seed(42)
np.random.seed(42)
print("\n✔  Environment ready.")

TensorFlow  : 2.19.0
NumPy       : 2.0.2
Eager mode  : True
GPU devices : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
CPU devices : 1

✔  Environment ready.


---
# Lecture 1: TensorFlow Execution Models and Performance Basics

## 1.1 What is TensorFlow?

TensorFlow is an open-source end-to-end ML framework by Google Brain (released 2015). It operates on **tensors** — multi-dimensional typed arrays — and supports:
- CPU, GPU, and TPU execution
- Automatic differentiation via `tf.GradientTape`
- High-level Keras API and low-level tensor operations
- Deployment via TF Serving, TF Lite, and TF.js

### The TensorFlow Abstraction Stack

```
[ Your Model / Application Code            ]
[ Keras High-Level API                     ]
[ tf low-level ops (tf.nn, tf.math, …)     ]
[ TF Runtime — C++ / XLA compiler          ]
[ Hardware — CPU / GPU / TPU               ]
```

In [2]:
# ── 1.1  Tensors: The Core Abstraction ────────────────────────────────

# ── Rank 0: Scalar ──
scalar = tf.constant(42, dtype=tf.int32)
print(f"Scalar  : value={scalar.numpy():>4}  shape={scalar.shape}  dtype={scalar.dtype.name}")

# ── Rank 1: Vector ──
vector = tf.constant([1.0, 2.0, 3.0])
print(f"Vector  : {vector.numpy()}  shape={vector.shape}")

# ── Rank 2: Matrix ──
matrix = tf.constant([[1, 2], [3, 4]], dtype=tf.float32)
print(f"Matrix  :\n{matrix.numpy()}  shape={matrix.shape}")

# ── Rank 3: Batch of matrices ──
tensor3d = tf.zeros((2, 3, 4))   # 2 matrices of size 3×4
print(f"3D Tensor shape: {tensor3d.shape}  (batch=2, rows=3, cols=4)")

# ── Tensors are IMMUTABLE ──
try:
    scalar[0] = 99
except Exception as e:
    print(f"\nTensors are immutable: {type(e).__name__}")

# ── tf.Variable: mutable, trainable state ──
weights = tf.Variable([[0.1, 0.2], [0.3, 0.4]], name='W')
weights.assign_add(tf.ones_like(weights) * 0.1)
print(f"\ntf.Variable after assign_add:\n{weights.numpy()}")

# ── NumPy interoperability ──
np_arr = np.array([10., 20., 30.])
tf_tensor = tf.constant(np_arr)
print(f"\nNumPy → TF : {tf_tensor}")
print(f"TF → NumPy : {tf_tensor.numpy()}")

Scalar  : value=  42  shape=()  dtype=int32
Vector  : [1. 2. 3.]  shape=(3,)
Matrix  :
[[1. 2.]
 [3. 4.]]  shape=(2, 2)
3D Tensor shape: (2, 3, 4)  (batch=2, rows=3, cols=4)

Tensors are immutable: TypeError

tf.Variable after assign_add:
[[0.2 0.3]
 [0.4 0.5]]

NumPy → TF : [10. 20. 30.]
TF → NumPy : [10. 20. 30.]


In [3]:
# ── 1.2  Core TensorFlow Operations ───────────────────────────────────

a = tf.constant([[1., 2.], [3., 4.]])
b = tf.constant([[5., 6.], [7., 8.]])

print("Element-wise add  (a + b):")
print(tf.add(a, b).numpy(), "\n")

print("Element-wise mul  (a * b):")
print(tf.multiply(a, b).numpy(), "\n")

print("Matrix multiply   (a @ b):")
print(tf.matmul(a, b).numpy(), "\n")

print("Reductions:")
print(f"  tf.reduce_sum(a)         = {tf.reduce_sum(a).numpy()}")
print(f"  tf.reduce_mean(a, axis=0) = {tf.reduce_mean(a, axis=0).numpy()}")
print(f"  tf.reduce_max(a)          = {tf.reduce_max(a).numpy()}")

print("\nReshaping:")
flat  = tf.reshape(a, [-1])          # flatten
col   = tf.reshape(a, [4, 1])        # column vector
print(f"  flatten  : {flat.numpy()}")
print(f"  column   : {col.numpy().T}")

print("\nActivations:")
x_act = tf.constant([-2., -1., 0., 1., 2.])
print(f"  Input   : {x_act.numpy()}")
print(f"  ReLU    : {tf.nn.relu(x_act).numpy()}")
print(f"  Sigmoid : {tf.nn.sigmoid(x_act).numpy().round(3)}")
print(f"  Softmax : {tf.nn.softmax(x_act).numpy().round(3)}")

Element-wise add  (a + b):
[[ 6.  8.]
 [10. 12.]] 

Element-wise mul  (a * b):
[[ 5. 12.]
 [21. 32.]] 

Matrix multiply   (a @ b):
[[19. 22.]
 [43. 50.]] 

Reductions:
  tf.reduce_sum(a)         = 10.0
  tf.reduce_mean(a, axis=0) = [2. 3.]
  tf.reduce_max(a)          = 4.0

Reshaping:
  flatten  : [1. 2. 3. 4.]
  column   : [[1. 2. 3. 4.]]

Activations:
  Input   : [-2. -1.  0.  1.  2.]
  ReLU    : [0. 0. 0. 1. 2.]
  Sigmoid : [0.119 0.269 0.5   0.731 0.881]
  Softmax : [0.012 0.032 0.086 0.234 0.636]


## 1.2 Automatic Differentiation with tf.GradientTape

`tf.GradientTape` records all operations executed inside it. When you call `tape.gradient(target, sources)`, it uses the recorded computation graph to compute **partial derivatives** automatically via reverse-mode autodiff (backpropagation).

In [4]:
# ── 1.3  Automatic Differentiation ────────────────────────────────────

# ── Scalar gradient: d/dx (x² + 3x + 2) at x = 3.0 ──
# Analytical: f'(x) = 2x + 3  →  f'(3) = 9
x = tf.Variable(3.0)
with tf.GradientTape() as tape:
    y = x**2 + 3*x + 2
dydx = tape.gradient(y, x)

print("f(x) = x² + 3x + 2  at x = 3")
print(f"  f(3)  = {y.numpy():.1f}")
print(f"  f'(3) = {dydx.numpy():.1f}  (expected: 9.0)  ✔")

# ── Multi-variable gradient ──
x1 = tf.Variable(2.0)
x2 = tf.Variable(3.0)
with tf.GradientTape() as tape:
    z = x1**2 * x2 + x2**3  # dz/dx1 = 2*x1*x2 = 12,  dz/dx2 = x1² + 3*x2² = 31
dz_dx1, dz_dx2 = tape.gradient(z, [x1, x2])
print(f"\nz = x1²·x2 + x2³  at x1=2, x2=3")
print(f"  dz/dx1 = {dz_dx1.numpy():.1f}  (expected: 12.0)  ✔")
print(f"  dz/dx2 = {dz_dx2.numpy():.1f}  (expected: 31.0)  ✔")

# ── Persistent tape (multi-call) ──
x = tf.Variable(3.0)
with tf.GradientTape(persistent=True) as tape:
    f  = x**3
    g  = x**2
df = tape.gradient(f, x)   # 3x² = 27
dg = tape.gradient(g, x)   # 2x  = 6
del tape
print(f"\nPersistent tape: df/dx={df.numpy():.1f} (expected 27), dg/dx={dg.numpy():.1f} (expected 6)  ✔")

f(x) = x² + 3x + 2  at x = 3
  f(3)  = 20.0
  f'(3) = 9.0  (expected: 9.0)  ✔

z = x1²·x2 + x2³  at x1=2, x2=3
  dz/dx1 = 12.0  (expected: 12.0)  ✔
  dz/dx2 = 31.0  (expected: 31.0)  ✔

Persistent tape: df/dx=27.0 (expected 27), dg/dx=6.0 (expected 6)  ✔


In [5]:
# ── 1.4  Mini Training Loop: Linear Regression ────────────────────────
#
# Goal: learn y = 2x + 1  from noisy samples

np.random.seed(42)
x_train = tf.constant(np.linspace(-3, 3, 50), dtype=tf.float32)
y_train = tf.constant(2.0 * x_train + 1.0 + np.random.randn(50) * 0.5, dtype=tf.float32)

W = tf.Variable(0.0, name='weight')
b = tf.Variable(0.0, name='bias')
optimizer = tf.optimizers.SGD(learning_rate=0.05)

losses = []
for step in range(200):
    with tf.GradientTape() as tape:
        y_pred = W * x_train + b
        loss   = tf.reduce_mean(tf.square(y_pred - y_train))
    grads = tape.gradient(loss, [W, b])
    optimizer.apply_gradients(zip(grads, [W, b]))
    losses.append(loss.numpy())

print(f"After 200 steps:")
print(f"  W = {W.numpy():.4f}  (target: 2.0)")
print(f"  b = {b.numpy():.4f}  (target: 1.0)")
print(f"  Final loss = {losses[-1]:.5f}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.scatter(x_train.numpy(), y_train.numpy(), s=20, alpha=0.6, label='Noisy data')
x_line = np.linspace(-3, 3, 100)
ax1.plot(x_line, W.numpy() * x_line + b.numpy(), 'r-', lw=2, label=f'Learned: y={W.numpy():.2f}x+{b.numpy():.2f}')
ax1.plot(x_line, 2*x_line + 1, 'g--', lw=1.5, label='True: y=2x+1')
ax1.legend(); ax1.set_title('Linear Regression via GradientTape')
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.grid(alpha=0.3)

ax2.semilogy(losses, color='#002060', lw=2)
ax2.set_title('Training Loss (MSE)'); ax2.set_xlabel('Step'); ax2.set_ylabel('Loss (log scale)')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('lec1_regression.png', dpi=100, bbox_inches='tight')
plt.show()
print("Figure saved.")

After 200 steps:
  W = 1.9517  (target: 2.0)
  b = 0.8873  (target: 1.0)
  Final loss = 0.20629
Figure saved.


## 1.3 Eager vs. Graph Execution

| Mode | How it works | When to use |
|------|-------------|-------------|
| **Eager** (default in TF2) | Ops run immediately in Python, like NumPy | Development, debugging |
| **Graph** (`@tf.function`) | TF traces the Python code once, builds a C++ graph, executes it | Production training & inference |

The graph mode wins because:
1. **No Python overhead** between ops
2. **Grappler optimizations** (constant folding, dead code elimination)
3. **Operator fusion** when combined with XLA

In [6]:
# ── 1.5  Eager vs Graph Execution Benchmark ────────────────────────────

def heavy_computation(x):
    """A compute-heavy sequence of ops."""
    for _ in range(10):
        x = tf.matmul(x, x)
        x = tf.nn.relu(x)
    return x

# Compiled (graph) version
heavy_graph = tf.function(heavy_computation)

data = tf.random.normal([32, 32])

# Warm-up: triggers graph tracing on first call
_ = heavy_graph(data)

N = 20

# Time eager
t0 = time.perf_counter()
for _ in range(N):
    _ = heavy_computation(data)
eager_ms = (time.perf_counter() - t0) / N * 1000

# Time graph
t0 = time.perf_counter()
for _ in range(N):
    _ = heavy_graph(data)
graph_ms = (time.perf_counter() - t0) / N * 1000

speedup = eager_ms / graph_ms
print(f"Eager execution  : {eager_ms:.3f} ms / call")
print(f"Graph execution  : {graph_ms:.3f} ms / call")
print(f"Speedup          : {speedup:.2f}x")

# Plot
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(['Eager\n(Python overhead)', 'Graph\n(@tf.function)'],
              [eager_ms, graph_ms],
              color=['#C00000', '#002060'], width=0.5, edgecolor='white')
for bar, val in zip(bars, [eager_ms, graph_ms]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
            f'{val:.2f} ms', ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_ylabel('Time per call (ms)', fontsize=12)
ax.set_title(f'Eager vs. Graph: {speedup:.1f}× speedup from @tf.function', fontsize=12)
ax.set_ylim(0, max(eager_ms, graph_ms) * 1.3)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('lec1_eager_vs_graph.png', dpi=100, bbox_inches='tight')
plt.show()

Eager execution  : 1.531 ms / call
Graph execution  : 0.785 ms / call
Speedup          : 1.95x


In [7]:
# ── 1.6  Inspecting the Computation Graph ─────────────────────────────

@tf.function
def forward_pass(x, w, b):
    """A two-layer forward pass."""
    h   = tf.nn.relu(tf.matmul(x, w) + b)   # hidden layer
    out = tf.reduce_mean(h)                  # scalar output
    return out

# Get the concrete function for a specific input signature
x_spec = tf.TensorSpec(shape=[None, 4], dtype=tf.float32)
w_spec = tf.TensorSpec(shape=[4, 8],    dtype=tf.float32)
b_spec = tf.TensorSpec(shape=[8],       dtype=tf.float32)

cf    = forward_pass.get_concrete_function(x_spec, w_spec, b_spec)
graph = cf.graph

print(f"{'Operation type':30s}  Node name")
print("-" * 60)
for op in graph.get_operations():
    print(f"  {op.type:28s}  {op.name}")

print(f"\nTotal ops   : {len(graph.get_operations())}")
print(f"Graph inputs : {[t.name for t in cf.inputs]}")
print(f"Graph outputs: {[t.name for t in cf.outputs]}")

Operation type                  Node name
------------------------------------------------------------
  Placeholder                   x
  Placeholder                   w
  Placeholder                   b
  MatMul                        MatMul
  AddV2                         add
  Relu                          Relu
  Const                         Const
  Mean                          Mean
  Identity                      Identity

Total ops   : 9
Graph inputs : ['x:0', 'w:0', 'b:0']
Graph outputs: ['Identity:0']


---
# Lecture 2: Optimizing Input Pipelines with tf.data

## 2.1 Why Input Pipelines Matter

Modern GPUs can process thousands of images per second. Without pipeline optimization:
- GPU finishes batch N → **sits idle** waiting for CPU to prepare batch N+1
- 40–70% of expensive GPU time is **wasted on I/O wait**

**Key idea:** Overlap CPU preprocessing with GPU training using **prefetching**.

```
NAIVE:    [Load][Preprocess][Train] [Load][Preprocess][Train] ...
                                   ^ GPU idle here ^

OPTIMIZED:[Load+Preprocess] [Train]
                     [Load+Preprocess] [Train]   ← zero idle time
```

In [8]:
# ── 2.1  Creating tf.data Datasets ─────────────────────────────────────

# ── Method 1: from in-memory NumPy arrays ──
images = np.random.randn(100, 14, 14, 1).astype(np.float32)
labels = np.random.randint(0, 10, 100).astype(np.int32)

ds_numpy = tf.data.Dataset.from_tensor_slices((images, labels))
print(f"from_tensor_slices → {ds_numpy.element_spec}")

# ── Method 2: from a Python generator ──
def data_generator():
    for i in range(30):
        yield np.random.randn(14, 14, 1).astype(np.float32), np.int32(i % 10)

ds_gen = tf.data.Dataset.from_generator(
    data_generator,
    output_signature=(
        tf.TensorSpec(shape=(14, 14, 1), dtype=tf.float32),
        tf.TensorSpec(shape=(),          dtype=tf.int32)
    )
)
print(f"from_generator      → {ds_gen.element_spec}")

# ── Method 3: range / synthetic ──
ds_range = tf.data.Dataset.range(100)
print(f"Dataset.range(100)  → {ds_range.element_spec}")

# ── Peek at elements ──
print("\nFirst 3 samples from ds_numpy:")
for img, lbl in ds_numpy.take(3):
    print(f"  image shape: {img.shape}  label: {lbl.numpy()}")

from_tensor_slices → (TensorSpec(shape=(14, 14, 1), dtype=tf.float32, name=None), TensorSpec(shape=(), dtype=tf.int32, name=None))
from_generator      → (TensorSpec(shape=(14, 14, 1), dtype=tf.float32, name=None), TensorSpec(shape=(), dtype=tf.int32, name=None))
Dataset.range(100)  → TensorSpec(shape=(), dtype=tf.int64, name=None)

First 3 samples from ds_numpy:
  image shape: (14, 14, 1)  label: 7
  image shape: (14, 14, 1)  label: 9
  image shape: (14, 14, 1)  label: 5


In [9]:
# ── 2.2  Core Dataset Transformations ──────────────────────────────────

# ── .map() : element-wise transformation ──
@tf.function
def preprocess(image, label):
    """Normalize to [-1, 1] and apply random noise augmentation."""
    image  = tf.cast(image, tf.float32) / 127.5 - 1.0
    image  = image + tf.random.normal(tf.shape(image), stddev=0.01)
    image  = tf.clip_by_value(image, -1.0, 1.0)
    return image, label

# Build the optimized pipeline step by step
ds = (
    tf.data.Dataset.from_tensor_slices((images, labels))
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)   # parallel CPU workers
    .shuffle(buffer_size=200, reshuffle_each_iteration=True) # randomise order
    .batch(16, drop_remainder=True)                          # group into batches
    .prefetch(tf.data.AUTOTUNE)                              # CPU prepares next batch while GPU trains
)

print("Pipeline element spec:")
for spec in ds.element_spec:
    print(f"  {spec}")

for img_b, lbl_b in ds.take(1):
    print(f"\nBatch → images: {img_b.shape}  labels: {lbl_b.shape}")
    print(f"  image value range : [{img_b.numpy().min():.3f}, {img_b.numpy().max():.3f}]")

Pipeline element spec:
  TensorSpec(shape=(16, 14, 14, 1), dtype=tf.float32, name=None)
  TensorSpec(shape=(16,), dtype=tf.int32, name=None)

Batch → images: (16, 14, 14, 1)  labels: (16,)
  image value range : [-1.000, -0.954]


In [10]:
# ── 2.3  .cache(): Eliminate Redundant Preprocessing ──────────────────
#
# Without .cache(): map() runs on EVERY epoch
# With    .cache(): map() runs ONCE; results stored in RAM (or file)

N_ITEMS = 200
ds_raw  = tf.data.Dataset.range(N_ITEMS)

def preprocess_int(x):
    return tf.cast(x, tf.float32) * 2.0

ds_no_cache = ds_raw.map(preprocess_int).batch(20)
ds_cached   = ds_raw.map(preprocess_int).cache().batch(20)

def time_epochs(ds, n_epochs=4, name=''):
    times = []
    for ep in range(n_epochs):
        t0 = time.perf_counter()
        for _ in ds: pass
        times.append((time.perf_counter() - t0) * 1000)
    print(f"  {name:25s}: {[f'{t:.1f}ms' for t in times]}")
    return times

print("Epoch times over 4 epochs (cache fills on epoch 1):")
t_no = time_epochs(ds_no_cache, name='Without .cache()')
t_c  = time_epochs(ds_cached,   name='With .cache()')

print(f"\nEpoch 2 speedup from .cache(): {t_no[1]/max(t_c[1], 0.001):.1f}×")

Epoch times over 4 epochs (cache fills on epoch 1):
  Without .cache()         : ['16.5ms', '11.2ms', '11.5ms', '11.1ms']
  With .cache()            : ['22.8ms', '3.1ms', '4.2ms', '3.2ms']

Epoch 2 speedup from .cache(): 3.7×


In [11]:
# ── 2.4  Parallel Preprocessing: num_parallel_calls ────────────────────

N = 400
x_data = np.random.randn(N, 14, 14, 1).astype(np.float32)
y_data = np.random.randint(0, 10, N).astype(np.int32)

@tf.function
def augment(img, lbl):
    img = img + tf.random.normal(tf.shape(img), stddev=0.05)
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    img = tf.clip_by_value(img, -3.0, 3.0)
    return img, lbl

# Sequential map
ds_seq = (tf.data.Dataset.from_tensor_slices((x_data, y_data))
          .map(augment, num_parallel_calls=1).batch(32).prefetch(1))

# Parallel map (AUTOTUNE picks optimal worker count)
ds_par = (tf.data.Dataset.from_tensor_slices((x_data, y_data))
          .map(augment, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE))

def time_pipeline(ds, reps=3):
    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        for _ in ds: pass
        times.append((time.perf_counter() - t0) * 1000)
    return np.mean(times)

t_seq = time_pipeline(ds_seq)
t_par = time_pipeline(ds_par)
print(f"Sequential map (workers=1)  : {t_seq:.1f} ms")
print(f"Parallel map (AUTOTUNE)     : {t_par:.1f} ms")
print(f"Parallelism speedup         : {t_seq/max(t_par,0.001):.2f}×")

Sequential map (workers=1)  : 100.2 ms
Parallel map (AUTOTUNE)     : 78.2 ms
Parallelism speedup         : 1.28×


In [12]:
# ── 2.5  TFRecord: The Optimal Disk Storage Format ─────────────────────
#
# Why TFRecord?
#  • Sequential reads → saturates disk bandwidth
#  • No per-file overhead like JPEG/PNG directories
#  • Native to TF → no Python decode overhead

tmpdir   = tempfile.mkdtemp()
tfr_path = os.path.join(tmpdir, 'demo.tfrecord')

# ── Write TFRecord ──
N_RECORDS = 100
with tf.io.TFRecordWriter(tfr_path) as writer:
    for i in range(N_RECORDS):
        img   = np.random.randint(0, 255, (14, 14, 1), dtype=np.uint8)
        lbl   = i % 10
        feature = {
            'image':  tf.train.Feature(bytes_list=tf.train.BytesList(value=[img.astype(np.float32).tobytes()])),
            'label':  tf.train.Feature(int64_list=tf.train.Int64List(value=[lbl])),
            'height': tf.train.Feature(int64_list=tf.train.Int64List(value=[14])),
            'width':  tf.train.Feature(int64_list=tf.train.Int64List(value=[14])),
        }
        ex = tf.train.Example(features=tf.train.Features(feature=feature))
        writer.write(ex.SerializeToString())

print(f"Written {N_RECORDS} records  →  {os.path.getsize(tfr_path)/1024:.1f} KB")

# ── Read TFRecord ──
feature_desc = {
    'image':  tf.io.FixedLenFeature([], tf.string),
    'label':  tf.io.FixedLenFeature([], tf.int64),
    'height': tf.io.FixedLenFeature([], tf.int64),
    'width':  tf.io.FixedLenFeature([], tf.int64),
}

def parse_example(serialized):
    parsed = tf.io.parse_single_example(serialized, feature_desc)
    image  = tf.io.decode_raw(parsed['image'], tf.float32)
    image  = tf.reshape(image, [14, 14, 1])
    return image, parsed['label']

ds_tfr = (
    tf.data.TFRecordDataset(tfr_path)
    .map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(200)
    .batch(16)
    .prefetch(tf.data.AUTOTUNE)
)

for img_b, lbl_b in ds_tfr.take(1):
    print(f"TFRecord batch → images: {img_b.shape}  labels[:5]: {lbl_b.numpy()[:5]}")

shutil.rmtree(tmpdir)
print("\nTFRecord pipeline verified ✔")

Written 100 records  →  85.1 KB
TFRecord batch → images: (16, 14, 14, 1)  labels[:5]: [0 1 3 3 8]

TFRecord pipeline verified ✔


In [13]:
# ── 2.6  Recommended Pipeline Order — Full Demonstration ──────────────
#
#   source → .cache() → .shuffle() → .map(AUTOTUNE) → .batch() → .prefetch(AUTOTUNE)

x_all = np.random.randn(400, 14, 14, 1).astype(np.float32)
y_all = np.random.randint(0, 10, 400).astype(np.int32)

@tf.function
def augment_v2(img, lbl):
    img = img + tf.random.normal(tf.shape(img), stddev=0.03)
    return tf.clip_by_value(img, -2.0, 2.0), lbl

optimal_ds = (
    tf.data.Dataset.from_tensor_slices((x_all, y_all))
    .cache()                                                   # ① cache raw data
    .shuffle(buffer_size=400, reshuffle_each_iteration=True)   # ② shuffle
    .map(augment_v2, num_parallel_calls=tf.data.AUTOTUNE)      # ③ parallel augmentation
    .batch(32, drop_remainder=True)                            # ④ batch
    .prefetch(tf.data.AUTOTUNE)                                # ⑤ prefetch
)

count = 0
t0 = time.perf_counter()
for img_b, lbl_b in optimal_ds:
    count += 1
elapsed_ms = (time.perf_counter() - t0) * 1000

print(f"Optimal pipeline: {count} batches of 32 in {elapsed_ms:.1f} ms")
print(f"Throughput: {400 / (elapsed_ms/1000):,.0f} samples/sec")

print("\nPipeline optimization summary:")
opts = [
    ('.cache()',                    'Stores preprocessed data in RAM → skip re-reading disk on epoch 2+'),
    ('.shuffle(large_buffer)',      'Larger buffer → more random order → better generalization'),
    ('.map(AUTOTUNE)',              'Uses all available CPU cores for preprocessing in parallel'),
    ('.batch(N, drop_remainder)',   'Groups N samples → vector ops faster than scalars'),
    ('.prefetch(AUTOTUNE)',         'CPU prepares batch N+1 while GPU trains on batch N'),
]
for name, benefit in opts:
    print(f"  {name:35s}  →  {benefit}")

Optimal pipeline: 12 batches of 32 in 39.6 ms
Throughput: 10,110 samples/sec

Pipeline optimization summary:
  .cache()                             →  Stores preprocessed data in RAM → skip re-reading disk on epoch 2+
  .shuffle(large_buffer)               →  Larger buffer → more random order → better generalization
  .map(AUTOTUNE)                       →  Uses all available CPU cores for preprocessing in parallel
  .batch(N, drop_remainder)            →  Groups N samples → vector ops faster than scalars
  .prefetch(AUTOTUNE)                  →  CPU prepares batch N+1 while GPU trains on batch N


---
# Lecture 3: Graph Execution and TensorFlow Optimization Techniques

## 3.1 @tf.function Deep Dive: Tracing and Caching

When you decorate a Python function with `@tf.function`, TensorFlow:
1. **Traces** the function: runs the Python code once to record operations into a `tf.Graph`
2. **Caches** the resulting `ConcreteFunction` keyed on (dtype, shape) of inputs
3. **Executes** the C++ graph directly on subsequent calls with matching signatures

In [14]:
# ── 3.1  Tracing and Caching Behaviour ────────────────────────────────

trace_log = []

@tf.function
def traced_fn(x):
    trace_log.append(f"Traced: dtype={x.dtype.name}, shape={x.shape}")
    tf.print("[tf.print inside graph] shape:", tf.shape(x))   # runs every call
    return tf.reduce_sum(x ** 2)

print("=== Call 1: float32 shape [3] — NEW trace ===")
r1 = traced_fn(tf.constant([1.0, 2.0, 3.0]))

print("\n=== Call 2: float32 shape [3] — CACHED graph ===")
r2 = traced_fn(tf.constant([4.0, 5.0, 6.0]))

print("\n=== Call 3: float32 shape [4] — NEW trace (shape changed) ===")
r3 = traced_fn(tf.constant([1.0, 2.0, 3.0, 4.0]))

print("\n=== Call 4: float64 — NEW trace (dtype changed) ===")
r4 = traced_fn(tf.constant([1.0, 2.0, 3.0], dtype=tf.float64))

print("\n=== Call 5: float32 shape [3] — CACHED graph (original sig) ===")
r5 = traced_fn(tf.constant([7.0, 8.0, 9.0]))

print(f"\nTotal Python-level traces (should be 3): {len(trace_log)}")
for msg in trace_log:
    print(f"  {msg}")

print("\nKey insight: tf.print runs EVERY call (it's in the graph).")
print("             trace_log only grows DURING TRACING (Python phase).")

=== Call 1: float32 shape [3] — NEW trace ===
[tf.print inside graph] shape: [3]

=== Call 2: float32 shape [3] — CACHED graph ===
[tf.print inside graph] shape: [3]

=== Call 3: float32 shape [4] — NEW trace (shape changed) ===
[tf.print inside graph] shape: [4]

=== Call 4: float64 — NEW trace (dtype changed) ===
[tf.print inside graph] shape: [3]

=== Call 5: float32 shape [3] — CACHED graph (original sig) ===
[tf.print inside graph] shape: [3]

Total Python-level traces (should be 3): 3
  Traced: dtype=float32, shape=(3,)
  Traced: dtype=float32, shape=(4,)
  Traced: dtype=float64, shape=(3,)

Key insight: tf.print runs EVERY call (it's in the graph).
             trace_log only grows DURING TRACING (Python phase).


In [15]:
# ── 3.2  input_signature: Prevent Unnecessary Retracing ────────────────

@tf.function(
    input_signature=[
        tf.TensorSpec(shape=[None, None], dtype=tf.float32)  # any 2D float32
    ]
)
def stable_fn(x):
    """Works with ANY 2D float32 input — compiled exactly once."""
    return tf.reduce_mean(x, axis=-1)

stable_trace_count = [0]

shapes = [(3, 4), (10, 20), (1, 100), (64, 128), (256, 512)]
print("Calling stable_fn with different shapes (all reuse the SAME graph):")
for shape in shapes:
    data = tf.random.normal(shape)
    out  = stable_fn(data)
    print(f"  Input {str(shape):12s} → Output shape: {out.shape}")

# Contrast: without input_signature
retrace_log2 = []

@tf.function
def unstable_fn(x):
    retrace_log2.append(x.shape)
    return tf.reduce_mean(x, axis=-1)

for shape in shapes:
    unstable_fn(tf.random.normal(shape))

print(f"\nWithout input_signature: {len(retrace_log2)} traces (one per shape)")
print(f"With    input_signature: exactly 1 trace")

Calling stable_fn with different shapes (all reuse the SAME graph):
  Input (3, 4)       → Output shape: (3,)
  Input (10, 20)     → Output shape: (10,)
  Input (1, 100)     → Output shape: (1,)
  Input (64, 128)    → Output shape: (64,)
  Input (256, 512)   → Output shape: (256,)



Without input_signature: 5 traces (one per shape)
With    input_signature: exactly 1 trace


In [16]:
# ── 3.3  AutoGraph: Python Control Flow → TF Graph Ops ────────────────
#
# AutoGraph converts Python control flow into TF graph ops:
#   Python if    →  tf.cond
#   Python for   →  tf.while_loop
#   Python while →  tf.while_loop
#
# NOTE: In Jupyter, AutoGraph can read function source automatically.
#       The explicit forms below work in ALL execution contexts.

# ── Pattern 1: if → tf.cond ──────────────────────────────────────────
@tf.function
def conditional_activation(x, threshold):
    """Returns relu(x) if max(x) > threshold, else zeros.
    AutoGraph equivalent: if tf.reduce_max(x) > threshold: ...
    """
    return tf.cond(
        tf.reduce_max(x) > threshold,
        true_fn=lambda: tf.nn.relu(x),
        false_fn=lambda: tf.zeros_like(x)
    )

# ── Pattern 2: for-range → tf.while_loop ─────────────────────────────
@tf.function
def power_series(x, n_terms):
    """Sum: x^0 + x^1 + ... + x^(n_terms-1).
    AutoGraph equivalent: for k in tf.range(n_terms): total += x**k
    """
    def cond(k, acc): return k < n_terms
    def body(k, acc): return k + 1, acc + x ** tf.cast(k, tf.float32)
    _, result = tf.while_loop(cond, body,
                              loop_vars=[tf.constant(0), tf.zeros_like(x)])
    return result

# ── Pattern 3: while → tf.while_loop ─────────────────────────────────
@tf.function
def grow_until(x, threshold):
    """Multiply x by 1.5 until sum > threshold.
    AutoGraph equivalent: while tf.reduce_sum(x) < threshold: x *= 1.5
    """
    count = tf.constant(0)
    def cond(cur_x, c): return tf.reduce_sum(cur_x) < threshold
    def body(cur_x, c): return cur_x * 1.5, c + 1
    x_out, count_out = tf.while_loop(cond, body, loop_vars=[x, count])
    return x_out, count_out

# ── Tests ──
pos = tf.constant([0.5, 1.0, -0.5])
neg = tf.constant([-1.0, -0.5, -0.1])
print(f"Pattern 1 — conditional_activation:")
print(f"  pos input → {conditional_activation(pos, 0.3).numpy()}  (relu applied)")
print(f"  neg input → {conditional_activation(neg, 0.3).numpy()}  (all zeros)")

x_ps = tf.constant([0.5])
ps   = power_series(x_ps, tf.constant(5))
print(f"\nPattern 2 — power_series([0.5], 5 terms):")
print(f"  result = {ps.numpy()[0]:.4f}  (expected 1.9375 = 1+0.5+0.25+0.125+0.0625)")

x_w, n_iters = grow_until(tf.constant([1.0]), tf.constant(10.0))
print(f"\nPattern 3 — grow_until(1.0, threshold=10):")
print(f"  {n_iters.numpy()} iterations,  final value = {x_w.numpy()[0]:.3f}")
print("\nAll AutoGraph patterns verified ✔")


Pattern 1 — conditional_activation:
  pos input → [0.5 1.  0. ]  (relu applied)
  neg input → [0. 0. 0.]  (all zeros)

Pattern 2 — power_series([0.5], 5 terms):
  result = 1.9375  (expected 1.9375 = 1+0.5+0.25+0.125+0.0625)

Pattern 3 — grow_until(1.0, threshold=10):
  6 iterations,  final value = 11.391

All AutoGraph patterns verified ✔


In [17]:
# ── 3.4  Full Custom Training Loop with @tf.function ──────────────────

def build_cnn(input_shape=(14, 14, 1), n_classes=10):
    return tf.keras.Sequential([
        tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same',
                               input_shape=input_shape),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(n_classes)
    ])

model     = build_cnn()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
loss_fn   = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

@tf.function
def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        logits = model(x_batch, training=True)
        loss   = loss_fn(y_batch, logits)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

@tf.function
def eval_step(x_batch, y_batch):
    logits  = model(x_batch, training=False)
    preds   = tf.argmax(logits, axis=-1, output_type=tf.int32)
    correct = tf.reduce_sum(tf.cast(preds == y_batch, tf.float32))
    loss    = loss_fn(y_batch, logits)
    return loss, correct

# Synthetic dataset
N_TRAIN, N_VAL = 300, 60
x_tr = np.random.randn(N_TRAIN, 14, 14, 1).astype(np.float32)
y_tr = np.random.randint(0, 10, N_TRAIN).astype(np.int32)
x_va = np.random.randn(N_VAL,   14, 14, 1).astype(np.float32)
y_va = np.random.randint(0, 10, N_VAL).astype(np.int32)

train_ds = (tf.data.Dataset.from_tensor_slices((x_tr, y_tr))
            .shuffle(300).batch(32).prefetch(tf.data.AUTOTUNE))
val_ds   = (tf.data.Dataset.from_tensor_slices((x_va, y_va))
            .batch(32).prefetch(tf.data.AUTOTUNE))

history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
N_EPOCHS = 4

for epoch in range(N_EPOCHS):
    tr_losses = []
    for xb, yb in train_ds:
        l = train_step(xb, yb)
        tr_losses.append(l.numpy())

    va_losses, correct_total = [], 0.0
    for xb, yb in val_ds:
        vl, vc = eval_step(xb, yb)
        va_losses.append(vl.numpy()); correct_total += vc.numpy()

    tr_l = np.mean(tr_losses)
    va_l = np.mean(va_losses)
    va_a = correct_total / N_VAL
    history['train_loss'].append(tr_l)
    history['val_loss'].append(va_l)
    history['val_acc'].append(va_a)
    print(f"Epoch {epoch+1}/{N_EPOCHS}  train_loss={tr_l:.4f}  val_loss={va_l:.4f}  val_acc={va_a:.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(history['train_loss'], 'b-o', label='Train Loss')
ax1.plot(history['val_loss'],   'r-o', label='Val Loss')
ax1.set_title('Loss Curves (Custom Training Loop)')
ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(history['val_acc'], 'g-o')
ax2.set_title('Validation Accuracy'); ax2.grid(alpha=0.3)
ax2.set_ylim(0, 0.5)
plt.tight_layout()
plt.savefig('lec3_training.png', dpi=100, bbox_inches='tight')
plt.show()

Epoch 1/4  train_loss=2.3130  val_loss=2.2852  val_acc=0.100
Epoch 2/4  train_loss=2.2936  val_loss=2.2920  val_acc=0.117
Epoch 3/4  train_loss=2.2922  val_loss=2.3000  val_acc=0.117
Epoch 4/4  train_loss=2.2797  val_loss=2.3041  val_acc=0.117


In [18]:
# ── 3.5  Grappler Optimizations: Constant Folding ─────────────────────
#
# TF's Grappler optimizer folds constant expressions at COMPILE TIME.
# Arithmetic on tf.constant values is pre-computed — never executed at runtime.

@tf.function
def with_constant_fold():
    a = tf.constant(2.0)
    b = tf.constant(3.0)
    c = a * b            # Grappler: 6.0 (compile-time)
    d = c + 10.0         # Grappler: 16.0 (compile-time)
    e = d ** 2           # Grappler: 256.0 (compile-time)
    return e

result = with_constant_fold()
print(f"Constant-folded: 2 × 3 = 6 → 6 + 10 = 16 → 16² = {result.numpy():.1f}  ✔")

# ── 3.6  Operator Fusion ──────────────────────────────────────────────
print("\nOperator Fusion (concept):")
print("  BEFORE fusion: Conv2D kernel → [write HBM] → BiasAdd → [write HBM] → ReLU")
print("  AFTER  fusion: Conv2D+BiasAdd+ReLU in ONE kernel — zero intermediate HBM writes")
print("  Memory bandwidth savings can be 30-60% for dense layers")

# Fusion via XLA (jit_compile=True)
@tf.function(jit_compile=True)
def fused_ops(x, w, b):
    h = tf.nn.conv2d(x, w, strides=1, padding='SAME')
    h = tf.nn.bias_add(h, b)
    return tf.nn.relu(h)

@tf.function(jit_compile=False)
def unfused_ops(x, w, b):
    h = tf.nn.conv2d(x, w, strides=1, padding='SAME')
    h = tf.nn.bias_add(h, b)
    return tf.nn.relu(h)

x_c = tf.random.normal([4, 16, 16, 4])
w_c = tf.random.normal([3, 3, 4, 8])
b_c = tf.zeros([8])
_ = fused_ops(x_c, w_c, b_c); _ = unfused_ops(x_c, w_c, b_c)  # warm-up

REPS = 20
t0 = time.perf_counter()
for _ in range(REPS): unfused_ops(x_c, w_c, b_c)
t_unf = (time.perf_counter()-t0)/REPS*1000

t0 = time.perf_counter()
for _ in range(REPS): fused_ops(x_c, w_c, b_c)
t_fus = (time.perf_counter()-t0)/REPS*1000

print(f"\n  Unfused Conv+Bias+ReLU : {t_unf:.3f} ms")
print(f"  XLA-fused              : {t_fus:.3f} ms")
print(f"  Speedup                : {t_unf/max(t_fus,0.001):.2f}×")

Constant-folded: 2 × 3 = 6 → 6 + 10 = 16 → 16² = 256.0  ✔

Operator Fusion (concept):
  BEFORE fusion: Conv2D kernel → [write HBM] → BiasAdd → [write HBM] → ReLU
  AFTER  fusion: Conv2D+BiasAdd+ReLU in ONE kernel — zero intermediate HBM writes
  Memory bandwidth savings can be 30-60% for dense layers

  Unfused Conv+Bias+ReLU : 0.634 ms
  XLA-fused              : 0.448 ms
  Speedup                : 1.42×


In [19]:
# ── 3.7  Mixed Precision: float16 Compute + float32 Accumulation ───────
#
# GPU Tensor Cores can run float16 matmul at 2–8× the speed of float32.
# Key rules:
#   • Compute (matmul, conv): use float16
#   • Loss / weight update:   keep float32 for numerical stability
#   • Use tf.keras.mixed_precision.set_global_policy('mixed_float16')

data_f32 = tf.random.normal([128, 128], dtype=tf.float32)
data_f16 = tf.cast(data_f32, tf.float16)

@tf.function
def matmul_f32(x): return tf.matmul(x, tf.transpose(x))

@tf.function
def matmul_f16(x): return tf.matmul(x, tf.transpose(x))

# Warm-up
_ = matmul_f32(data_f32); _ = matmul_f16(data_f16)

REPS = 30
t0 = time.perf_counter()
for _ in range(REPS): matmul_f32(data_f32)
t_f32 = (time.perf_counter()-t0)/REPS*1000

t0 = time.perf_counter()
for _ in range(REPS): matmul_f16(data_f16)
t_f16 = (time.perf_counter()-t0)/REPS*1000

print("Mixed Precision Results (128×128 matmul):")
print(f"  float32 : {t_f32:.3f} ms")
print(f"  float16 : {t_f16:.3f} ms")
print(f"  Speedup : {t_f32/max(t_f16,0.001):.2f}×  (GPU speedup much larger — 2–8× on Tensor Cores)")

print("\nMemory footprint:")
n = 128 * 128
print(f"  float32 tensor (128×128): {n*4 / 1024:.1f} KB")
print(f"  float16 tensor (128×128): {n*2 / 1024:.1f} KB  (50% saving)")

print("\nTo enable globally:")
print("  tf.keras.mixed_precision.set_global_policy('mixed_float16')")
print("  # Then use model.compile() as usual — Keras handles scaling automatically")

Mixed Precision Results (128×128 matmul):
  float32 : 0.481 ms
  float16 : 0.488 ms
  Speedup : 0.98×  (GPU speedup much larger — 2–8× on Tensor Cores)

Memory footprint:
  float32 tensor (128×128): 64.0 KB
  float16 tensor (128×128): 32.0 KB  (50% saving)

To enable globally:
  tf.keras.mixed_precision.set_global_policy('mixed_float16')
  # Then use model.compile() as usual — Keras handles scaling automatically


---
# Lecture 4: Introduction to XLA Compilation (Demo)

## 4.1 What is XLA?

**XLA (Accelerated Linear Algebra)** is a domain-specific compiler for linear algebra computations. It works in three stages:

1. **Frontend:** TF computation graph → HLO IR (High-Level Operations)
2. **Optimization:** Constant folding, operator fusion, algebraic simplification, layout assignment
3. **Backend:** HLO → native code for CPU / GPU / TPU

**Primary win:** Operator fusion eliminates intermediate memory writes to HBM (High-Bandwidth Memory). For a `Conv → BiasAdd → ReLU` chain, XLA generates a **single kernel** instead of three.

In [20]:
# ── 4.1  Three Ways to Enable XLA ─────────────────────────────────────

print("Method 1: Per-function  (most targeted)")
print("""
  @tf.function(jit_compile=True)
  def train_step(x, y):
      with tf.GradientTape() as tape:
          loss = loss_fn(y, model(x))
      ...
""")

print("Method 2: Keras compile()  (compiles train + eval + predict)")
print("""
  model.compile(
      optimizer='adam',
      loss=loss_fn,
      jit_compile=True          # <-- one line change
  )
""")

print("Method 3: Environment variable  (global, affects all graphs)")
print("""
  # Before launching Python:
  export TF_XLA_FLAGS='--tf_xla_auto_jit=2'
""")

print("Rule of thumb:")
print("  Use Method 1 for hot loops (train_step, eval_step)")
print("  Use Method 2 when using model.fit()")
print("  Use Method 3 for experimentation / profiling")

Method 1: Per-function  (most targeted)

  @tf.function(jit_compile=True)
  def train_step(x, y):
      with tf.GradientTape() as tape:
          loss = loss_fn(y, model(x))
      ...

Method 2: Keras compile()  (compiles train + eval + predict)

  model.compile(
      optimizer='adam',
      loss=loss_fn,
      jit_compile=True          # <-- one line change
  )

Method 3: Environment variable  (global, affects all graphs)

  # Before launching Python:
  export TF_XLA_FLAGS='--tf_xla_auto_jit=2'

Rule of thumb:
  Use Method 1 for hot loops (train_step, eval_step)
  Use Method 2 when using model.fit()
  Use Method 3 for experimentation / profiling


In [21]:
# ── 4.2  XLA vs Standard: Benchmark ───────────────────────────────────

def build_bench_model():
    inp = tf.keras.Input(shape=(8, 8, 3))
    x = tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu')(inp)
    x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    out = tf.keras.layers.Dense(10)(x)
    return tf.keras.Model(inp, out)

model_xla   = build_bench_model()
model_noxla = build_bench_model()
model_xla.set_weights(model_noxla.get_weights())  # identical weights

loss_fn   = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
opt_xla   = tf.keras.optimizers.Adam(1e-3)
opt_noxla = tf.keras.optimizers.Adam(1e-3)

@tf.function(jit_compile=True)
def step_xla(x, y):
    with tf.GradientTape() as tape:
        loss = loss_fn(y, model_xla(x, training=True))
    grads = tape.gradient(loss, model_xla.trainable_variables)
    opt_xla.apply_gradients(zip(grads, model_xla.trainable_variables))
    return loss

@tf.function(jit_compile=False)
def step_std(x, y):
    with tf.GradientTape() as tape:
        loss = loss_fn(y, model_noxla(x, training=True))
    grads = tape.gradient(loss, model_noxla.trainable_variables)
    opt_noxla.apply_gradients(zip(grads, model_noxla.trainable_variables))
    return loss

BATCH, N_STEPS = 16, 20
x_b = tf.random.normal([BATCH, 8, 8, 3])
y_b = tf.random.uniform([BATCH], 0, 10, dtype=tf.int32)

print("Warming up (XLA compiles on first call)...")
_ = step_xla(x_b, y_b); _ = step_std(x_b, y_b)
print("Done.")

t0 = time.perf_counter()
for _ in range(N_STEPS): step_std(x_b, y_b)
t_std = (time.perf_counter()-t0)/N_STEPS*1000

t0 = time.perf_counter()
for _ in range(N_STEPS): step_xla(x_b, y_b)
t_xla = (time.perf_counter()-t0)/N_STEPS*1000

speedup = t_std / max(t_xla, 0.001)
print(f"\nStandard @tf.function : {t_std:.3f} ms/step")
print(f"XLA jit_compile=True  : {t_xla:.3f} ms/step")
print(f"Speedup               : {speedup:.2f}×")

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(['Standard\n@tf.function', 'XLA\njit_compile=True'],
              [t_std, t_xla], color=['#C00000', '#375623'], width=0.45)
for bar, val in zip(bars, [t_std, t_xla]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
            f'{val:.2f}ms', ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('Time per step (ms)', fontsize=12)
ax.set_title(f'XLA Speedup: {speedup:.2f}×', fontsize=13)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('lec4_xla_speedup.png', dpi=100, bbox_inches='tight')
plt.show()

Warming up (XLA compiles on first call)...
Done.

Standard @tf.function : 2.436 ms/step
XLA jit_compile=True  : 0.803 ms/step
Speedup               : 3.03×


In [22]:
# ── 4.3  XLA via model.compile(jit_compile=True) ──────────────────────

model_kxla = build_bench_model()
model_kxla.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
    jit_compile=True                # ← XLA for ALL steps: train, eval, predict
)

x_k = np.random.randn(128, 8, 8, 3).astype(np.float32)
y_k = np.random.randint(0, 10, 128).astype(np.int32)

print("Training with jit_compile=True:")
h = model_kxla.fit(x_k, y_k, batch_size=32, epochs=3,
                   validation_split=0.2, verbose=1)
print(f"\nFinal train loss   : {h.history['loss'][-1]:.4f}")
print(f"Final val loss     : {h.history['val_loss'][-1]:.4f}")

Training with jit_compile=True:
Epoch 1/3
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 578ms/step - loss: 2.3148 - sparse_categorical_accuracy: 0.0588 - val_loss: 2.3222 - val_sparse_categorical_accuracy: 0.1154
Epoch 2/3
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 2.2972 - sparse_categorical_accuracy: 0.0882 - val_loss: 2.3273 - val_sparse_categorical_accuracy: 0.1538
Epoch 3/3
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 2.2917 - sparse_categorical_accuracy: 0.1373 - val_loss: 2.3305 - val_sparse_categorical_accuracy: 0.1154

Final train loss   : 2.2917
Final val loss     : 2.3305


In [23]:
# ── 4.4  First-Call Compilation Cost vs Steady-State Speedup ──────────

@tf.function(jit_compile=True)
def xla_bench(x):
    for _ in range(5):
        x = tf.nn.relu(x + tf.cast(tf.shape(x)[0], tf.float32) * 0.001)
    return tf.reduce_mean(x)

@tf.function(jit_compile=False)
def std_bench(x):
    for _ in range(5):
        x = tf.nn.relu(x + tf.cast(tf.shape(x)[0], tf.float32) * 0.001)
    return tf.reduce_mean(x)

d = tf.random.normal([64, 64])

# First call (XLA compiles)
t0 = time.perf_counter()
_ = xla_bench(d)
first_xla_ms = (time.perf_counter()-t0)*1000

t0 = time.perf_counter()
_ = std_bench(d)
first_std_ms = (time.perf_counter()-t0)*1000

# Steady-state
N = 50
t0 = time.perf_counter()
for _ in range(N): std_bench(d)
steady_std_ms = (time.perf_counter()-t0)/N*1000

t0 = time.perf_counter()
for _ in range(N): xla_bench(d)
steady_xla_ms = (time.perf_counter()-t0)/N*1000

print(f"First call  (compilation overhead):")
print(f"  Standard : {first_std_ms:.2f} ms")
print(f"  XLA      : {first_xla_ms:.2f} ms")
print(f"\nSteady-state ({N} calls avg):")
print(f"  Standard : {steady_std_ms:.3f} ms/call")
print(f"  XLA      : {steady_xla_ms:.3f} ms/call")
extra_cost = first_xla_ms - first_std_ms
per_step_saving = steady_std_ms - steady_xla_ms
break_even = extra_cost / max(per_step_saving, 0.001)
print(f"\nXLA break-even: ~{int(break_even)} steps  (worthwhile for long training runs)")

First call  (compilation overhead):
  Standard : 115.42 ms
  XLA      : 377.21 ms

Steady-state (50 calls avg):
  Standard : 0.677 ms/call
  XLA      : 0.468 ms/call

XLA break-even: ~1255 steps  (worthwhile for long training runs)


In [24]:
# ── 4.5  XLA Benefit Scales with Problem Size ──────────────────────────

@tf.function(jit_compile=True)
def xla_chain(x):
    x = tf.nn.relu(x + 0.01)
    x = tf.nn.sigmoid(x - 0.5)
    x = tf.nn.tanh(x)
    return tf.reduce_sum(x)

@tf.function(jit_compile=False)
def std_chain(x):
    x = tf.nn.relu(x + 0.01)
    x = tf.nn.sigmoid(x - 0.5)
    x = tf.nn.tanh(x)
    return tf.reduce_sum(x)

sizes = [32, 128, 512, 1024]
xla_ts, std_ts, speedups = [], [], []
REPS = 30

for sz in sizes:
    d_s = tf.random.normal([sz, sz])
    _ = xla_chain(d_s); _ = std_chain(d_s)   # warm-up

    t0 = time.perf_counter()
    for _ in range(REPS): std_chain(d_s)
    ts = (time.perf_counter()-t0)/REPS*1000

    t0 = time.perf_counter()
    for _ in range(REPS): xla_chain(d_s)
    tx = (time.perf_counter()-t0)/REPS*1000

    std_ts.append(ts); xla_ts.append(tx); speedups.append(ts/max(tx,0.001))
    print(f"Size {sz:5d}×{sz}: std={ts:.3f}ms  xla={tx:.3f}ms  speedup={ts/max(tx,0.001):.2f}×")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(sizes, std_ts, 'ro-', lw=2, ms=7, label='Standard')
ax1.plot(sizes, xla_ts, 'gs-', lw=2, ms=7, label='XLA')
ax1.set_xlabel('Matrix size N'); ax1.set_ylabel('ms / call')
ax1.set_title('Execution Time vs Problem Size')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.bar(range(len(sizes)), speedups, color='#002060', edgecolor='white')
ax2.axhline(1.0, color='red', ls='--', lw=1.5)
ax2.set_xticks(range(len(sizes)))
ax2.set_xticklabels([f'{s}×{s}' for s in sizes])
for i, s in enumerate(speedups):
    ax2.text(i, s*1.02, f'{s:.2f}×', ha='center', fontsize=10, fontweight='bold')
ax2.set_xlabel('Matrix size'); ax2.set_ylabel('XLA Speedup')
ax2.set_title('XLA Speedup Grows with Problem Size')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('lec4_xla_scaling.png', dpi=100, bbox_inches='tight')
plt.show()
print("\nKey insight: XLA fusion benefit grows with tensor size (more memory bandwidth saved).")

Size    32×32: std=0.673ms  xla=0.630ms  speedup=1.07×
Size   128×128: std=0.532ms  xla=0.322ms  speedup=1.65×
Size   512×512: std=0.591ms  xla=0.330ms  speedup=1.79×
Size  1024×1024: std=0.539ms  xla=0.360ms  speedup=1.50×

Key insight: XLA fusion benefit grows with tensor size (more memory bandwidth saved).


---
# Lecture 5: Model Interoperability – SavedModel vs ONNX

## 5.1 Why Model Serialization Matters

A model trained in Python must be deployed in:
- A **C++ production server** (TF Serving)
- A **mobile app** (TF Lite / Core ML)
- A **cross-framework inference engine** (ONNX Runtime, TensorRT)
- A **different ML framework** (TF → ONNX → PyTorch)

## Format Comparison

| | `.keras` | `model.export()` SavedModel | ONNX |
|--|--|--|--|
| Use | Keras→Keras | TF Serving / TF Lite / tf2onnx | Cross-framework, ORT, TensorRT |
| Portability | Low | Medium | **High** |
| Contains | Full Keras config | TF signatures only | Platform-neutral protobuf |
| Convert from | `model.save()` | `model.export()` | tf2onnx CLI |


In [25]:
# ── 5.1  Build and Train the Demo Model ───────────────────────────────

def build_demo_model():
    inp = tf.keras.Input(shape=(20,), name='features')
    x   = tf.keras.layers.Dense(64, activation='relu', name='hidden1')(inp)
    x   = tf.keras.layers.BatchNormalization(name='bn')(x)
    x   = tf.keras.layers.Dense(32, activation='relu', name='hidden2')(x)
    out = tf.keras.layers.Dense(5,  activation='softmax', name='predictions')(x)
    return tf.keras.Model(inp, out)

model = build_demo_model()
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()]
)
model.summary()

x_demo = np.random.randn(200, 20).astype(np.float32)
y_demo = np.random.randint(0, 5, 200).astype(np.int32)

model.fit(x_demo, y_demo, epochs=5, batch_size=32, verbose=0)
print("\nModel trained ✔")

# Reference predictions for later consistency checks
x_test  = np.random.randn(10, 20).astype(np.float32)
ref_preds = model.predict(x_test, verbose=0)
print(f"Reference predictions shape: {ref_preds.shape}")
print(f"Sample: {ref_preds[0].round(4)}  (sums to {ref_preds[0].sum():.4f})")

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ features (InputLayer)           │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden1 (Dense)                 │ (None, 64)             │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn (BatchNormalization)         │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,845 (15.02 KB)

 Trainable params: 3,717 (14.52 KB)

 Non-trainable params: 128 (512.00 B)


Model trained ✔
Reference predictions shape: (10, 5)
Sample: [0.0856 0.2763 0.2437 0.1741 0.2204]  (sums to 1.0000)


In [26]:
# ── 5.2  Keras Native Format: model.save() / load_model() ─────────────
#
# .keras format stores:
#   • Full model architecture (JSON config)
#   • Trained weights (HDF5 / zip)
#   • Optimizer state (for resuming training)
#   • Training config (loss, metrics)

tmpdir     = tempfile.mkdtemp()
keras_path = os.path.join(tmpdir, 'model.keras')

# Save
model.save(keras_path)
size_kb = os.path.getsize(keras_path) / 1024
print(f"Saved to '{keras_path}'  ({size_kb:.1f} KB)")

# Load
loaded_keras = tf.keras.models.load_model(keras_path)
preds_keras  = loaded_keras.predict(x_test, verbose=0)

match = np.allclose(ref_preds, preds_keras, atol=1e-5)
max_diff = np.max(np.abs(ref_preds - preds_keras))
print(f"Keras load → predictions match: {match}  (max diff: {max_diff:.2e})  ✔")

# Confirm optimizer state is preserved
print(f"\nOptimizer iterations: {loaded_keras.optimizer.iterations.numpy()}")
print(f"(non-zero → optimizer state was restored for training resumption)")

Saved to '/tmp/tmpbrjtbwur/model.keras'  (77.2 KB)
Keras load → predictions match: True  (max diff: 0.00e+00)  ✔

Optimizer iterations: 35
(non-zero → optimizer state was restored for training resumption)


In [27]:
# ── 5.3  TF SavedModel: model.export() ────────────────────────────────
#
# Use model.export() to create a TF SavedModel for:
#   • TF Serving (REST/gRPC API)
#   • TF Lite conversion
#   • tf2onnx conversion

export_path = os.path.join(tmpdir, 'saved_model')
model.export(export_path)

print("SavedModel directory structure:")
for root, dirs, files in os.walk(export_path):
    indent = '  ' * (root.replace(export_path, '').count(os.sep))
    print(f"{indent}📁 {os.path.basename(root)}/")
    for f in files:
        size = os.path.getsize(os.path.join(root, f))
        print(f"{indent}   📄 {f}  ({size/1024:.1f} KB)")

# Load via tf.saved_model.load()
loaded_sm = tf.saved_model.load(export_path)

# Keras 3 export exposes a .serve endpoint
sm_preds = loaded_sm.serve(tf.constant(x_test)).numpy()
sm_match = np.allclose(ref_preds, sm_preds, atol=1e-4)
print(f"\ntf.saved_model.load().serve() predictions match: {sm_match}  ✔")

Saved artifact at '/tmp/tmpbrjtbwur/saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 20), dtype=tf.float32, name='features')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  135167791283280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135165196981968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135165196981008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135165196978704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135165196978512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135165196980624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135165196980816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135165196981392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135165196976592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135165196978896: TensorSpec(shape=(), dtype=tf.resource, name=None)
SavedModel direct

In [30]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras

custom_path = os.path.join(tmpdir, "custom_model")

# Custom serving function: pre-processing + inference + post-processing
def serving_fn(features):
    features = tf.clip_by_value(features, -5.0, 5.0)
    probs = model(features, training=False)
    return {
        "class_id": tf.argmax(probs, axis=-1, output_type=tf.int32),
        "confidence": tf.reduce_max(probs, axis=-1),
        "probabilities": probs,
    }

# Keras 3 / official export path
archive = keras.export.ExportArchive()
archive.track(model)
archive.add_endpoint(
    name="serving_default",
    fn=serving_fn,
    input_signature=[
        tf.TensorSpec(shape=(None, 20), dtype=tf.float32, name="features")
    ],
)
archive.write_out(custom_path)

print(f"Saved with custom signature to: {custom_path}")

# Load and test
c_loaded = tf.saved_model.load(custom_path)
c_infer = c_loaded.signatures["serving_default"]

result = c_infer(features=tf.constant(x_test[:3], dtype=tf.float32))

print("\nCustom signature output:")
print(f"  class_ids     : {result['class_id'].numpy()}")
print(f"  confidence    : {result['confidence'].numpy().round(4)}")
print(f"  probabilities :\n    {result['probabilities'].numpy().round(4)}")

# Verify consistency with the serving logic (must compare against clipped inputs)
x_test_clipped = tf.clip_by_value(tf.constant(x_test, dtype=tf.float32), -5.0, 5.0)
ref_preds_clipped = model(x_test_clipped, training=False).numpy()

preds_custom = c_infer(features=tf.constant(x_test, dtype=tf.float32))["probabilities"].numpy()
match = np.allclose(ref_preds_clipped, preds_custom, atol=1e-4)

print(f"\nCustom sig predictions match clipped-model reference: {match}  ✔")

Saved artifact at '/tmp/tmpbrjtbwur/custom_model'. The following endpoints are available:

* Endpoint 'serving_default'
  features (POSITIONAL_OR_KEYWORD): TensorSpec(shape=(None, 20), dtype=tf.float32, name='features')
Output Type:
  Dict[['class_id', TensorSpec(shape=(None,), dtype=tf.int32, name=None)], ['confidence', TensorSpec(shape=(None,), dtype=tf.float32, name=None)], ['probabilities', TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)]]
Captures:
  135167791283280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135165196981968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135165196981008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135165196978704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135165196978512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135165196980624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135165196980816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135165196981392: TensorSpec(shape=(), dty

In [36]:
# ── 5.5  ONNX Export: TF → ONNX ───────────────────────────────────────
#
# ONNX (Open Neural Network Exchange) is an open format backed by
# Microsoft, Meta, AMD, Intel, NVIDIA, and others.
#
# Conversion pipeline:
#   model.export() → TF SavedModel → tf2onnx → model.onnx → ONNX Runtime
!pip install tf2onnx

import subprocess, sys

onnx_path = os.path.join(tmpdir, 'model.onnx')

cmd = [
    sys.executable, '-m', 'tf2onnx.convert',
    '--saved-model', export_path,
    '--output',      onnx_path,
    '--opset',       '17',           # ONNX opset version
]

print(f"Running: {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=True, text=True)

if os.path.exists(onnx_path):
    size_kb = os.path.getsize(onnx_path) / 1024
    print(f"\n✔  ONNX export successful!")
    print(f"   Path : {onnx_path}")
    print(f"   Size : {size_kb:.1f} KB")
else:
    print(f"Export stdout: {result.stdout[-300:]}")
    print(f"Export stderr: {result.stderr[-300:]}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 48.0 MB/s eta 0:00:00
Running: /usr/bin/python3 -m tf2onnx.convert --saved-model /tmp/tmpbrjtbwur/saved_model --output /tmp/tmpbrjtbwur/model.onnx --opset 17

✔  ONNX export successful!
   Path : /tmp/tmpbrjtbwur/model.onnx
   Size : 17.8 KB


In [37]:
# ── 5.6  Inspect the ONNX Graph ────────────────────────────────────────
!pip install onnx
import onnx

if os.path.exists(onnx_path):
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print("ONNX model is valid ✔")
    print(f"  IR version   : {onnx_model.ir_version}")
    print(f"  Opset        : {onnx_model.opset_import[0].version}")
    print(f"  Producer     : {onnx_model.producer_name}")
    print(f"  Total nodes  : {len(onnx_model.graph.node)}")

    # Inputs
    print("\nGraph inputs:")
    for inp in onnx_model.graph.input:
        shape = [d.dim_value if d.dim_value > 0 else '?'
                 for d in inp.type.tensor_type.shape.dim]
        print(f"  {inp.name:35s}  shape={shape}")

    # Outputs
    print("Graph outputs:")
    for out in onnx_model.graph.output:
        shape = [d.dim_value if d.dim_value > 0 else '?'
                 for d in out.type.tensor_type.shape.dim]
        print(f"  {out.name:35s}  shape={shape}")

    # Operation type histogram
    op_counts = {}
    for node in onnx_model.graph.node:
        op_counts[node.op_type] = op_counts.get(node.op_type, 0) + 1

    print("\nOperation counts:")
    for op, cnt in sorted(op_counts.items(), key=lambda x: -x[1]):
        bar = '█' * cnt
        print(f"  {op:20s} {bar} ({cnt})")
else:
    print("ONNX file not found — skipping inspection.")

ONNX model is valid ✔
  IR version   : 8
  Opset        : 17
  Producer     : tf2onnx
  Total nodes  : 11

Graph inputs:
  features                             shape=['?', 20]
Graph outputs:
  output_0                             shape=['?', 5]

Operation counts:
  Add                  ████ (4)
  MatMul               ███ (3)
  Relu                 ██ (2)
  Mul                  █ (1)
  Softmax              █ (1)


In [40]:
# ── 5.7  ONNX Runtime Inference ────────────────────────────────────────
!pip install onnxruntime

import onnxruntime as ort

if os.path.exists(onnx_path):
    # Create optimized inference session
    opts = ort.SessionOptions()
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    opts.intra_op_num_threads = 4

    session = ort.InferenceSession(
        onnx_path, opts,
        providers=['CPUExecutionProvider']
    )

    in_name  = session.get_inputs()[0].name
    out_name = session.get_outputs()[0].name
    print(f"ORT session ready")
    print(f"  Input  : {in_name}  {session.get_inputs()[0].shape}")
    print(f"  Output : {out_name}  {session.get_outputs()[0].shape}")
    print(f"  Providers: {session.get_providers()}")

    # Run inference
    ort_preds = session.run([out_name], {in_name: x_test})[0]
    max_diff  = np.max(np.abs(ref_preds - ort_preds))
    match     = np.allclose(ref_preds, ort_preds, atol=1e-4)

    print(f"\nNumerical consistency check:")
    print(f"  Predictions match (atol=1e-4): {match}")
    print(f"  Max absolute difference       : {max_diff:.2e}")

    print(f"\nSample output:")
    print(f"  TF Keras : {ref_preds[0].round(4)}")
    print(f"  ORT      : {ort_preds[0].round(4)}")
else:
    print("ONNX file unavailable.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 73.7 MB/s eta 0:00:00
ORT session ready
  Input  : features  ['unk__6', 20]
  Output : output_0  ['unk__7', 5]
  Providers: ['CPUExecutionProvider']

Numerical consistency check:
  Predictions match (atol=1e-4): True
  Max absolute difference       : 7.45e-08

Sample output:
  TF Keras : [0.0856 0.2763 0.2437 0.1741 0.2204]
  ORT      : [0.0856 0.2763 0.2437 0.1741 0.2204]


In [41]:
# ── 5.8  Latency Benchmark: TF Keras vs ONNX Runtime ──────────────────

if os.path.exists(onnx_path):
    N_RUNS   = 50
    x_single = np.random.randn(1, 20).astype(np.float32)   # single sample (latency)
    x_batch  = np.random.randn(32, 20).astype(np.float32)  # batch (throughput)

    x_tf_s = tf.constant(x_single)
    x_tf_b = tf.constant(x_batch)

    # ── Single-sample latency ──
    t0 = time.perf_counter()
    for _ in range(N_RUNS): model(x_tf_s, training=False)
    tf_s_ms = (time.perf_counter()-t0)/N_RUNS*1000

    t0 = time.perf_counter()
    for _ in range(N_RUNS): session.run([out_name], {in_name: x_single})
    ort_s_ms = (time.perf_counter()-t0)/N_RUNS*1000

    # ── Batch throughput ──
    t0 = time.perf_counter()
    for _ in range(N_RUNS): model(x_tf_b, training=False)
    tf_b_ms = (time.perf_counter()-t0)/N_RUNS*1000

    t0 = time.perf_counter()
    for _ in range(N_RUNS): session.run([out_name], {in_name: x_batch})
    ort_b_ms = (time.perf_counter()-t0)/N_RUNS*1000

    print(f"{'Scenario':25s}  {'TF Keras':>12s}  {'ONNX Runtime':>14s}  {'ORT speedup':>12s}")
    print("-" * 68)
    print(f"{'Single sample (bs=1)':25s}  {tf_s_ms:>10.3f}ms  {ort_s_ms:>12.3f}ms  {tf_s_ms/max(ort_s_ms,0.001):>10.2f}×")
    print(f"{'Batch (bs=32)':25s}  {tf_b_ms:>10.3f}ms  {ort_b_ms:>12.3f}ms  {tf_b_ms/max(ort_b_ms,0.001):>10.2f}×")

    tf_tp  = 32 / (tf_b_ms  / 1000)
    ort_tp = 32 / (ort_b_ms / 1000)
    print(f"\nThroughput (samples/sec):")
    print(f"  TF Keras       : {tf_tp:>8,.0f}")
    print(f"  ONNX Runtime   : {ort_tp:>8,.0f}")

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, title, vals in zip(
        axes,
        ['Single Sample Latency (ms)', 'Batch=32 Latency (ms)'],
        [(tf_s_ms, ort_s_ms), (tf_b_ms, ort_b_ms)]
    ):
        bars = ax.bar(['TF Keras', 'ONNX Runtime'], vals,
                      color=['#002060', '#ED7D31'], width=0.4, edgecolor='white')
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
                    f'{v:.3f}ms', ha='center', fontsize=11, fontweight='bold')
        ax.set_title(title); ax.set_ylabel('Time (ms)'); ax.grid(axis='y', alpha=0.3)

    plt.suptitle('TF Keras vs ONNX Runtime Inference Latency', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('lec5_latency.png', dpi=100, bbox_inches='tight')
    plt.show()
else:
    print("ONNX file unavailable — skipping benchmark.")

Scenario                       TF Keras    ONNX Runtime   ORT speedup
--------------------------------------------------------------------
Single sample (bs=1)            5.579ms         0.016ms      356.01×
Batch (bs=32)                   4.956ms         0.033ms      148.85×

Throughput (samples/sec):
  TF Keras       :    6,457
  ONNX Runtime   :  961,057


In [42]:
# ── 5.9  Decision Guide: When to Use Each Format ─────────────────────

guide = [
    (".keras",
     "Keras native (zip)",
     ["Resume training or fine-tuning",
      "Share model within Keras/TF ecosystem",
      "Preserve optimizer state + compile config"],
     "model.save('model.keras')"),

    ("SavedModel",
     "TF runtime protobuf",
     ["Deploy with TF Serving (REST/gRPC)",
      "Convert to TF Lite for mobile/edge",
      "Use as source for tf2onnx conversion",
      "Need custom serving signatures"],
     "model.export('saved_model/')"),

    ("ONNX",
     "Open cross-framework protobuf",
     ["Target ONNX Runtime (Windows/Linux/mobile)",
      "Optimize with NVIDIA TensorRT",
      "Optimize with Intel OpenVINO",
      "Deploy on Apple CoreML",
      "Interop with PyTorch / Scikit-learn models"],
     "tf2onnx.convert --saved-model ./sm --output model.onnx"),
]

for fmt, desc, when, how in guide:
    print(f"{'='*60}")
    print(f"  {fmt:15s}  [{desc}]")
    print(f"  How to create: {how}")
    print(f"  Use when:")
    for item in when:
        print(f"    ✔  {item}")
    print()

  .keras           [Keras native (zip)]
  How to create: model.save('model.keras')
  Use when:
    ✔  Resume training or fine-tuning
    ✔  Share model within Keras/TF ecosystem
    ✔  Preserve optimizer state + compile config

  SavedModel       [TF runtime protobuf]
  How to create: model.export('saved_model/')
  Use when:
    ✔  Deploy with TF Serving (REST/gRPC)
    ✔  Convert to TF Lite for mobile/edge
    ✔  Use as source for tf2onnx conversion
    ✔  Need custom serving signatures

  ONNX             [Open cross-framework protobuf]
  How to create: tf2onnx.convert --saved-model ./sm --output model.onnx
  Use when:
    ✔  Target ONNX Runtime (Windows/Linux/mobile)
    ✔  Optimize with NVIDIA TensorRT
    ✔  Optimize with Intel OpenVINO
    ✔  Deploy on Apple CoreML
    ✔  Interop with PyTorch / Scikit-learn models



In [43]:
# ── 5.10  End-to-End Production Pipeline ──────────────────────────────

print("Complete Train → Save → Export → Serve Pipeline")
print("=" * 50)

prod_tmp = tempfile.mkdtemp()
prod_dir = os.path.join(prod_tmp, 'production')
os.makedirs(prod_dir, exist_ok=True)

# Step 1: Train
print("\n[Step 1] Training...")
prod_model = build_demo_model()
prod_model.compile('adam', 'sparse_categorical_crossentropy',
                   metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])
prod_model.fit(x_demo, y_demo, epochs=5, batch_size=32, verbose=0)
print("  ✔  Model trained")

# Step 2: Evaluate
print("\n[Step 2] Evaluating...")
x_eval = np.random.randn(100, 20).astype(np.float32)
y_eval = np.random.randint(0, 5, 100).astype(np.int32)
loss_e, acc_e = prod_model.evaluate(x_eval, y_eval, verbose=0)
print(f"  ✔  loss={loss_e:.4f}  acc={acc_e:.4f}")

# Step 3: Save .keras (for retraining)
print("\n[Step 3] Saving .keras (for retraining)...")
os.makedirs(prod_dir, exist_ok=True)
keras_p = os.path.join(prod_dir, 'model.keras')
prod_model.save(keras_p)
print(f"  ✔  {keras_p}  ({os.path.getsize(keras_p)//1024} KB)")

# Step 4: Export SavedModel (for serving)
print("\n[Step 4] Exporting SavedModel (for TF Serving)...")
sm_p = os.path.join(prod_dir, 'saved_model')
prod_model.export(sm_p)
sm_size = sum(os.path.getsize(os.path.join(r,f))
              for r, _, fs in os.walk(sm_p) for f in fs) // 1024
print(f"  ✔  {sm_p}  ({sm_size} KB)")

# Step 5: Convert to ONNX
print("\n[Step 5] Converting to ONNX...")
onnx_p = os.path.join(prod_dir, 'model.onnx')
r = subprocess.run([
    sys.executable, '-m', 'tf2onnx.convert',
    '--saved-model', sm_p, '--output', onnx_p, '--opset', '17'
], capture_output=True, text=True)
if os.path.exists(onnx_p):
    print(f"  ✔  {onnx_p}  ({os.path.getsize(onnx_p)//1024} KB)")
else:
    print("  ⚠  ONNX conversion failed (see stderr above)")

# Step 6: Verify both serve the same results
print("\n[Step 6] Verifying consistency across all formats...")
x_serve = np.random.randn(5, 20).astype(np.float32)
preds_keras_ref = prod_model.predict(x_serve, verbose=0)
preds_keras = tf.keras.models.load_model(keras_p).predict(x_serve, verbose=0)
# Verify keras reload matches original
print(f"  .keras reload match : {np.allclose(preds_keras_ref, preds_keras, atol=1e-4)}")

if os.path.exists(onnx_p):
    sess_p = ort.InferenceSession(onnx_p, providers=['CPUExecutionProvider'])
    preds_ort = sess_p.run([sess_p.get_outputs()[0].name],
                           {sess_p.get_inputs()[0].name: x_serve})[0]
    print(f"  ONNX ORT vs .keras  : {np.allclose(preds_keras, preds_ort, atol=1e-4)}")

shutil.rmtree(prod_tmp)
print("\n✔  Pipeline complete. Temp files cleaned up.")

Complete Train → Save → Export → Serve Pipeline

[Step 1] Training...
  ✔  Model trained

[Step 2] Evaluating...
  ✔  loss=1.6287  acc=0.1600

[Step 3] Saving .keras (for retraining)...
  ✔  /tmp/tmp85u60vs4/production/model.keras  (77 KB)

[Step 4] Exporting SavedModel (for TF Serving)...
Saved artifact at '/tmp/tmp85u60vs4/production/saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 20), dtype=tf.float32, name='features')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  135163235346064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135163235340880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135163235345104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135163235347216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135163235347792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135163235346448: TensorSpec(shape=(), dtype=tf.resource, name=N

---
## Practice Exercises

### Exercise 1: Optimize a Slow tf.data Pipeline
The `slow_ds` below is intentionally unoptimized. Rewrite it as `fast_ds` applying every optimization you learned.

In [44]:
# ── Exercise 1 ─────────────────────────────────────────────────────────

x_ex = np.random.randn(400, 14, 14, 1).astype(np.float32)
y_ex = np.random.randint(0, 10, 400).astype(np.int32)

def normalize(img, lbl):
    return tf.cast(img, tf.float32) / 255.0, lbl

# ── SLOW pipeline ──
slow_ds = (
    tf.data.Dataset.from_tensor_slices((x_ex, y_ex))
    .map(normalize)            # single worker
    .shuffle(50)               # tiny buffer → poor randomness
    .batch(16)
    # no .prefetch()
)

t0 = time.perf_counter()
for _ in slow_ds: pass
slow_ms = (time.perf_counter()-t0)*1000
print(f"Slow pipeline : {slow_ms:.1f} ms")

# ── YOUR OPTIMIZED PIPELINE ── (solution provided for reference)
fast_ds = (
    tf.data.Dataset.from_tensor_slices((x_ex, y_ex))
    .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)    # ① parallel workers
    .cache()                                                 # ② cache after first epoch
    .shuffle(400, reshuffle_each_iteration=True)             # ③ full-size buffer
    .batch(32, drop_remainder=True)                          # ④ larger batch
    .prefetch(tf.data.AUTOTUNE)                              # ⑤ overlap with training
)

t0 = time.perf_counter()
for _ in fast_ds: pass
fast_ms = (time.perf_counter()-t0)*1000
print(f"Fast pipeline : {fast_ms:.1f} ms")
print(f"Speedup       : {slow_ms/max(fast_ms,0.001):.2f}×")

Slow pipeline : 28.7 ms
Fast pipeline : 30.6 ms
Speedup       : 0.94×


### Exercise 2: Count @tf.function Retraces

In [45]:
# ── Exercise 2 ─────────────────────────────────────────────────────────
# How many times does mystery_fn retrace?
# Predict before running, then verify.

retrace_events = []

@tf.function
def mystery_fn(x, scale):
    retrace_events.append({'dtype': x.dtype.name, 'shape': x.shape.as_list(),
                           'scale': scale})
    return tf.reduce_sum(x) * tf.cast(scale, x.dtype)

calls = [
    (tf.constant([1., 2., 3.]),                 tf.constant(2)),    # NEW  trace (float32[3])
    (tf.constant([4., 5., 6.]),                 tf.constant(3)),    # CACHED (float32[3])
    (tf.constant([1., 2.]),                     tf.constant(2)),    # NEW  trace (float32[2])
    (tf.constant([1., 2., 3.], dtype=tf.float64), tf.constant(2)), # NEW  trace (float64[3])
    (tf.constant([7., 8., 9.]),                 tf.constant(5)),    # CACHED (float32[3])
    (tf.constant([10., 11.]),                   tf.constant(1)),    # CACHED (float32[2])
]

for xv, sv in calls:
    mystery_fn(xv, sv)

print(f"Total retraces: {len(retrace_events)}  (should be 3)")
print("\nRetrace events:")
for ev in retrace_events:
    print(f"  dtype={ev['dtype']:8s}  shape={str(ev['shape']):10s}")

print("\nExplanation:")
print("  TF creates one ConcreteFunction per (dtype, shape) pair.")
print("  'scale' is a tf.constant → same dtype → no new trace.")
print("  Retraces: (float32[3]) + (float32[2]) + (float64[3]) = 3")

Total retraces: 3  (should be 3)

Retrace events:
  dtype=float32   shape=[3]       
  dtype=float32   shape=[2]       
  dtype=float64   shape=[3]       

Explanation:
  TF creates one ConcreteFunction per (dtype, shape) pair.
  'scale' is a tf.constant → same dtype → no new trace.
  Retraces: (float32[3]) + (float32[2]) + (float64[3]) = 3


### Exercise 3: XLA vs Standard — Ablation Study

In [46]:
# ── Exercise 3 ─────────────────────────────────────────────────────────
# Measure XLA benefit for increasingly complex operation chains.

def make_chain_fn(n_ops, use_xla):
    """Build a function with n_ops element-wise ops."""
    def fn(x):
        for _ in range(n_ops):
            x = tf.nn.relu(x + 0.01)
            x = tf.nn.sigmoid(x - 0.5)
        return tf.reduce_mean(x)
    return tf.function(fn, jit_compile=use_xla)

op_counts = [1, 3, 5, 10, 20]
DATA = tf.random.normal([128, 128])
REPS = 20

results = []
print(f"{'Ops':>6}  {'Standard':>12}  {'XLA':>12}  {'Speedup':>10}")
print("-" * 45)
for n in op_counts:
    fn_std = make_chain_fn(n, use_xla=False)
    fn_xla = make_chain_fn(n, use_xla=True)

    # Warm-up
    _ = fn_std(DATA); _ = fn_xla(DATA)

    t0 = time.perf_counter()
    for _ in range(REPS): fn_std(DATA)
    t_s = (time.perf_counter()-t0)/REPS*1000

    t0 = time.perf_counter()
    for _ in range(REPS): fn_xla(DATA)
    t_x = (time.perf_counter()-t0)/REPS*1000

    sp = t_s / max(t_x, 0.001)
    results.append((n, t_s, t_x, sp))
    print(f"{n:6d}  {t_s:10.3f}ms  {t_x:10.3f}ms  {sp:8.2f}×")

# Plot
ns, t_stds, t_xlas, sps = zip(*results)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(ns, t_stds, 'ro-', lw=2, ms=7, label='Standard')
ax1.plot(ns, t_xlas, 'gs-', lw=2, ms=7, label='XLA')
ax1.set_xlabel('Number of element-wise ops'); ax1.set_ylabel('ms / call')
ax1.set_title('Standard vs XLA: Execution Time'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.bar(ns, sps, color='#002060', edgecolor='white', width=1.5)
ax2.axhline(1.0, color='red', ls='--', lw=1.5, label='No gain baseline')
ax2.set_xlabel('Number of ops'); ax2.set_ylabel('XLA Speedup')
ax2.set_title('XLA Speedup Grows with Op Count (more fusion)'); ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('ex3_xla_ablation.png', dpi=100, bbox_inches='tight')
plt.show()
print("\nKey takeaway: More fusable ops → more XLA benefit.")

   Ops      Standard           XLA     Speedup
---------------------------------------------
     1       0.614ms       0.345ms      1.78×
     3       1.140ms       1.152ms      0.99×
     5       0.703ms       0.339ms      2.07×
    10       0.905ms       0.471ms      1.92×
    20       1.244ms       0.436ms      2.85×

Key takeaway: More fusable ops → more XLA benefit.


In [47]:
# ── Week 5 Summary Visualization ───────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Week 5: TensorFlow Acceleration — Key Concepts at a Glance',
             fontsize=14, fontweight='bold')

# 1. TF Stack
ax = axes[0, 0]
layers   = ['Application / Model', 'Keras High-Level API', 'tf Low-Level Ops',
             'TF Runtime + XLA Compiler', 'Hardware (CPU / GPU / TPU)']
widths   = [0.9, 0.78, 0.65, 0.78, 0.9]
col_list = ['#002060', '#1F4E79', '#2E6CA4', '#3A88BF', '#5EB3D4']
for i, (l, w, c) in enumerate(zip(layers, widths, col_list)):
    ax.barh(i, w, color=c, height=0.72, left=(1-w)/2)
    ax.text(0.5, i, l, ha='center', va='center', color='white', fontsize=8.5, fontweight='bold')
ax.set_xlim(0, 1); ax.set_ylim(-0.5, len(layers)-0.5)
ax.axis('off'); ax.set_title('TF Ecosystem Stack', fontweight='bold')

# 2. Execution mode speed (normalized)
ax = axes[0, 1]
cats = ['Eager\n(Python)', 'Graph\n(@tf.fn)', 'XLA\n(jit)']
vals = [1.0, 0.6, 0.38]
cols = ['#C00000', '#1F4E79', '#375623']
bars = ax.bar(cats, vals, color=cols, width=0.5, edgecolor='white')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.02, f'{v:.2f}×', ha='center',
            fontsize=11, fontweight='bold')
ax.set_ylabel('Relative time (lower = faster)')
ax.set_title('Execution Mode Comparison', fontweight='bold')
ax.set_ylim(0, 1.3); ax.grid(axis='y', alpha=0.3)

# 3. tf.data pipeline stages
ax = axes[0, 2]
stages   = ['Source', '.cache()', '.shuffle()', '.map()\n(AUTOTUNE)', '.batch()', '.prefetch()']
gains    = [1.0, 1.5, 1.0, 1.8, 1.0, 2.2]
bcolors  = ['#555', '#2E6CA4', '#3A88BF', '#C9A84C', '#3A88BF', '#375623']
ax.bar(range(len(stages)), gains, color=bcolors, edgecolor='white')
ax.set_xticks(range(len(stages))); ax.set_xticklabels(stages, fontsize=7.5)
ax.set_ylabel('Relative throughput improvement')
ax.set_title('tf.data Pipeline Optimization Gains', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# 4. XLA operator fusion
ax = axes[1, 0]
without = ['Conv2D', 'HBM write', 'BiasAdd', 'HBM write', 'ReLU', 'HBM write']
ax.barh(range(6), [1]*6, color='#C00000', alpha=0.75)
for i, lbl in enumerate(without):
    ax.text(0.5, i, lbl, ha='center', va='center', color='white', fontsize=8, fontweight='bold')
ax.barh([7], [3], color='#375623', alpha=0.9)
ax.text(1.5, 7, 'Fused Conv+Bias+ReLU  (1 kernel)', ha='center', va='center',
        color='white', fontsize=8, fontweight='bold')
ax.set_xlim(0, 4); ax.set_ylim(-0.5, 8)
ax.set_title('Operator Fusion (XLA)', fontweight='bold')
ax.axis('off')
red_p  = mpatches.Patch(color='#C00000', alpha=0.75, label='Without XLA (6 HBM writes)')
grn_p  = mpatches.Patch(color='#375623', alpha=0.9,  label='With XLA (0 HBM writes)')
ax.legend(handles=[red_p, grn_p], loc='lower right', fontsize=8)

# 5. Format comparison
ax = axes[1, 1]
fmts        = ['.keras', 'SavedModel', 'ONNX', 'TF Lite']
portability = [1, 2, 5, 3]
infer_speed = [2, 3, 5, 4]
xp = np.arange(len(fmts)); w2 = 0.35
ax.bar(xp - w2/2, portability, w2, label='Portability', color='#1F4E79', edgecolor='white')
ax.bar(xp + w2/2, infer_speed, w2, label='Inference Speed', color='#C9A84C', edgecolor='white')
ax.set_xticks(xp); ax.set_xticklabels(fmts, fontsize=9)
ax.set_ylabel('Score (1–5)'); ax.set_title('Model Format Comparison', fontweight='bold')
ax.legend(fontsize=9); ax.set_ylim(0, 6.5); ax.grid(axis='y', alpha=0.3)

# 6. Cheat sheet
ax = axes[1, 2]
ax.axis('off')
cheat = (
    "  Week 5 Quick-Reference Cheat Sheet\n"
    "  ─────────────────────────────────\n"
    "\n"
    "  Execution modes:\n"
    "    Eager → dev/debug only\n"
    "    @tf.function → production training\n"
    "    jit_compile=True → XLA speedup\n"
    "\n"
    "  tf.data golden order:\n"
    "    source → .cache() → .shuffle()\n"
    "    → .map(AUTOTUNE) → .batch()\n"
    "    → .prefetch(AUTOTUNE)\n"
    "\n"
    "  Serialization:\n"
    "    Stay in TF? → model.export()\n"
    "    Cross-framework? → ONNX\n"
    "    Mobile / edge? → TF Lite\n"
    "    Retrain? → model.save('.keras')"
)
ax.text(0.03, 0.97, cheat, transform=ax.transAxes,
        fontsize=9, va='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', fc='#EBF3FA', ec='#002060', lw=2))
ax.set_title('Quick Reference', fontweight='bold')

plt.tight_layout()
plt.savefig('week5_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print("Summary figure saved.")

Summary figure saved.


In [48]:
# ── Final Checklist ────────────────────────────────────────────────────

print("""
╔══════════════════════════════════════════════════════════════════════╗
║        Week 5: TensorFlow Acceleration — Notebook Complete           ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  Lecture 1: TF Execution Models & Performance Basics                 ║
║    ✔  Tensors, tf.Variable, NumPy interop                            ║
║    ✔  tf.GradientTape: scalar, multi-var, persistent tape            ║
║    ✔  GradientTape-based linear regression training loop             ║
║    ✔  Eager vs Graph benchmark with @tf.function                     ║
║    ✔  Computation graph inspection via get_concrete_function()        ║
║                                                                      ║
║  Lecture 2: Optimizing Input Pipelines with tf.data                  ║
║    ✔  Dataset creation: from_tensor_slices, from_generator, range    ║
║    ✔  .map(AUTOTUNE), .cache(), .shuffle(), .batch(), .prefetch()    ║
║    ✔  Parallel vs sequential map() benchmark                         ║
║    ✔  TFRecord write/read complete pipeline                          ║
║    ✔  Optimal pipeline order with throughput measurement             ║
║                                                                      ║
║  Lecture 3: Graph Execution & TF Optimization                        ║
║    ✔  @tf.function tracing and caching mechanics                     ║
║    ✔  input_signature to prevent retracing                           ║
║    ✔  AutoGraph: if→tf.cond, for→tf.while_loop, while→tf.while_loop ║
║    ✔  Full custom training loop with @tf.function                    ║
║    ✔  Grappler constant folding + XLA operator fusion demo           ║
║    ✔  Mixed precision: float16 vs float32 memory + speed             ║
║                                                                      ║
║  Lecture 4: XLA Compilation (Demo)                                   ║
║    ✔  Three methods: @tf.function, compile(), env var                ║
║    ✔  Benchmarked XLA speedup (training step)                        ║
║    ✔  Keras model.compile(jit_compile=True)                          ║
║    ✔  First-call compilation cost vs steady-state savings            ║
║    ✔  XLA benefit scales with problem size (ablation study)          ║
║                                                                      ║
║  Lecture 5: SavedModel vs ONNX                                       ║
║    ✔  model.save('.keras') + load_model() + consistency check        ║
║    ✔  model.export() → TF SavedModel + serve() inference             ║
║    ✔  Custom serving signatures with pre/post-processing             ║
║    ✔  tf2onnx CLI export pipeline                                    ║
║    ✔  ONNX model validation + graph inspection + op histogram        ║
║    ✔  ONNX Runtime inference + numerical consistency check           ║
║    ✔  TF vs ORT latency benchmark (single + batch)                   ║
║    ✔  Decision guide + complete production pipeline                  ║
║                                                                      ║
║  Exercises:                                                          ║
║    ✔  Ex 1: tf.data pipeline optimization                            ║
║    ✔  Ex 2: @tf.function retrace counting                            ║
║    ✔  Ex 3: XLA ablation study (op count vs speedup)                 ║
╚══════════════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════════════╗
║        Week 5: TensorFlow Acceleration — Notebook Complete           ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  Lecture 1: TF Execution Models & Performance Basics                 ║
║    ✔  Tensors, tf.Variable, NumPy interop                            ║
║    ✔  tf.GradientTape: scalar, multi-var, persistent tape            ║
║    ✔  GradientTape-based linear regression training loop             ║
║    ✔  Eager vs Graph benchmark with @tf.function                     ║
║    ✔  Computation graph inspection via get_concrete_function()        ║
║                                                                      ║
║  Lecture 2: Optimizing Input Pipelines with tf.data                  ║
║    ✔  Dataset creation: from_tensor_slices, from_generator, range    ║
║    ✔  .map(AUTOTUNE), .cache(), .shuffle(), .ba